In [1]:
!pip install gurobipy
import math
import os
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(params={
    "WLSACCESSID": "f42db593-9cf1-4f65-878f-3c76fce25504",
    "WLSSECRET": "d7406968-0567-4590-9af8-18b64756a74b",
    "LICENSEID": 2810930,
})

games = pd.read_csv("games.csv")
games["Date"] = pd.to_datetime(games["Date"], format="%a, %b %d, %Y")

teams = sorted(set(games["Home"]) | set(games["Visitor"]))
dates = sorted(games["Date"].unique())

print(f"{len(teams)} teams, {len(dates)} dates, {len(games)} games "
      f"({dates[0]:%b %d} – {dates[-1]:%b %d, %Y})")

Set parameter Username
Set parameter LicenseID to value 2778238
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2810930
Academic license 2810930 - for non-commercial use only - registered to cc___@columbia.edu


FileNotFoundError: [Errno 2] No such file or directory: 'games.csv'

## Question 1

For each team *i*, we print:
- (a) the dates i plays at home
- (b) how often i hosts each opponent j
- (c) how often i visits each opponent j's arena
- (d) the dates i plays away

In [ ]:
home_dates = {t: list(games.loc[games["Home"] == t, "Date"]) for t in teams}
away_dates = {t: list(games.loc[games["Visitor"] == t, "Date"]) for t in teams}

home_count = pd.crosstab(games["Home"], games["Visitor"]).reindex(index=teams, columns=teams, fill_value=0)
away_count = pd.crosstab(games["Visitor"], games["Home"]).reindex(index=teams, columns=teams, fill_value=0)

for t in teams:
    print(f"\n{t}")
    print(f"(a) home ({len(home_dates[t])}): " + ", ".join(f"{d:%b %d}" for d in home_dates[t]))
    print(f"(d) away ({len(away_dates[t])}): " + ", ".join(f"{d:%b %d}" for d in away_dates[t]))
    print(f"(b) hosts: " + ", ".join(f"{j[:3].upper()}={home_count.loc[t, j]}" for j in teams if j != t))
    print(f"(c) visits: " + ", ".join(f"{j[:3].upper()}={away_count.loc[t, j]}" for j in teams if j != t))


Atlanta Hawks
(a) home (10): Nov 03, Nov 07, Nov 15, Nov 17, Nov 19, Nov 23, Nov 27, Nov 28, Nov 29, Dec 25
(d) away (6): Nov 01, Nov 05, Nov 11, Nov 13, Nov 21, Dec 01
(b) hosts: BOS=1, BRO=0, CHI=1, CLE=0, DAL=1, DEN=0, GOL=1, HOU=0, LOS=1, MIA=1, MIL=1, NEW=1, PHI=0, PHO=1, TOR=1
(c) visits: BOS=0, BRO=1, CHI=1, CLE=1, DAL=0, DEN=1, GOL=0, HOU=1, LOS=0, MIA=0, MIL=0, NEW=0, PHI=1, PHO=0, TOR=0

Boston Celtics
(a) home (6): Nov 01, Nov 07, Nov 13, Nov 19, Nov 23, Nov 28
(d) away (10): Nov 03, Nov 05, Nov 11, Nov 15, Nov 17, Nov 21, Nov 27, Nov 29, Dec 01, Dec 25
(b) hosts: ATL=0, BRO=0, CHI=0, CLE=0, DAL=1, DEN=0, GOL=1, HOU=0, LOS=0, MIA=1, MIL=1, NEW=0, PHI=1, PHO=1, TOR=0
(c) visits: ATL=1, BRO=1, CHI=1, CLE=1, DAL=0, DEN=1, GOL=1, HOU=1, LOS=1, MIA=0, MIL=0, NEW=1, PHI=0, PHO=0, TOR=1

Brooklyn Nets
(a) home (7): Nov 01, Nov 05, Nov 11, Nov 13, Nov 21, Nov 29, Dec 01
(d) away (9): Nov 03, Nov 07, Nov 15, Nov 17, Nov 19, Nov 23, Nov 27, Nov 28, Dec 25
(b) hosts: ATL=1, BOS=1, CHI

## Question 2

**Variables:**
- $x_{ijd} = 1$ — if team $i$ hosts team $j$ on date $d$, for $i \neq j$, $d \in D$

**Constraints:**
- (e) $\sum_{j \neq i} x_{ijd} = \mathbf{1}[d \in H_i]$ — *i* plays at home on dates provided
- (f) $\sum_{j \neq i} x_{jid} = \mathbf{1}[d \in A_i]$ — *i* plays away on dates provided
- (g) $\sum_d x_{ijd} = c^H_{ij}$ — home matchup counts preserved
- (h) $\sum_d x_{jid} = c^A_{ij}$ — away matchup counts preserved

In [ ]:
def base_model(name):
    m = gp.Model(name, env=env)
    m.Params.LogToConsole = 0

    x = m.addVars(
        [(i, j, d) for i in teams for j in teams if i != j for d in dates],
        vtype=GRB.BINARY, name="x",
    )

    H = {t: set(home_dates[t]) for t in teams}
    A = {t: set(away_dates[t]) for t in teams}

    for i in teams:
        for d in dates:
            # (e) home dates
            m.addConstr(gp.quicksum(x[i, j, d] for j in teams if j != i) == int(d in H[i]))
            # (f) away dates
            m.addConstr(gp.quicksum(x[j, i, d] for j in teams if j != i) == int(d in A[i]))

    for i in teams:
        for j in teams:
            if i == j:
                continue
            # (g) and (h) matchup counts
            m.addConstr(gp.quicksum(x[i, j, d] for d in dates) == home_count.loc[i, j])
            m.addConstr(gp.quicksum(x[j, i, d] for d in dates) == away_count.loc[i, j])

    return m, x


def schedule_from_x(x):
    rows = [
        {"Date": d.strftime("%a, %b %d, %Y"), "Home": i, "Visitor": j}
        for d in dates for i in teams for j in teams
        if i != j and x[i, j, d].X > 0.5
    ]
    return pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)


m, x = base_model("Q2_feasibility")
m.optimize()
print(schedule_from_x(x).head(10).to_string(index=False))

Set parameter LogToConsole to value 0
             Date                Home               Visitor
Fri, Nov 07, 2025      Boston Celtics Golden State Warriors
Fri, Nov 07, 2025    Dallas Mavericks    Los Angeles Lakers
Fri, Nov 07, 2025      Denver Nuggets       Houston Rockets
Fri, Nov 07, 2025     Milwaukee Bucks         Brooklyn Nets
Fri, Nov 07, 2025  Philadelphia 76ers       New York Knicks
Fri, Nov 07, 2025     Toronto Raptors            Miami Heat
Fri, Nov 07, 2025       Atlanta Hawks          Phoenix Suns
Fri, Nov 07, 2025 Cleveland Cavaliers         Chicago Bulls
Fri, Nov 21, 2025        Phoenix Suns      Dallas Mavericks
Fri, Nov 21, 2025     Milwaukee Bucks            Miami Heat


## Question 3 — Timezone Constraint

For each team $t$ and consecutive 3 games $(d_1, d_2, d_3)$:
$$|tz(t, d_2) - tz(t, d_1)| + |tz(t, d_3) - tz(t, d_2)| \le 3.$$

Because constraints (e), (f) put each team in exactly one arena per game date,
$tz(t, d)$ is linear in $x$:
$$tz(t, d) = \mathrm{tz}(\text{home}_t)\,\sum_{j} x_{tjd} \;+\; \sum_{j} \mathrm{tz}(\text{home}_j)\,x_{jtd}.$$

We linearize each absolute value with a non-negative auxiliary variable.

In [ ]:
arena_tz = {
    "TD Garden": -5, "Madison Square Garden": -5, "Barclays Center": -5,
    "Wells Fargo Center": -5, "Scotiabank Arena": -5, "State Farm Arena": -5,
    "Rocket Mortgage FieldHouse": -5, "Kaseya Center": -5,
    "United Center": -6, "Fiserv Forum": -6, "Toyota Center": -6,
    "American Airlines Center": -6,
    "Ball Arena": -7, "Footprint Center": -7,
    "Chase Center": -8, "Crypto.com Arena": -8,
}
team_arena = dict(zip(games["Home"], games["Arena"]))
team_tz = {t: arena_tz[team_arena[t]] for t in teams}

# Each team's full game sequence in date order, with the arena it played at
def team_sequence(team, schedule):
    arena_of_home = dict(zip(schedule["Home"], schedule["Arena"]))
    home = schedule.loc[schedule["Home"]    == team, ["Date", "Arena"]]
    away = schedule.loc[schedule["Visitor"] == team, ["Date", "Home"]].rename(columns={"Home": "Host"})
    away["Arena"] = away["Host"].map(arena_of_home)
    seq = pd.concat([home, away[["Date", "Arena"]]]).sort_values("Date")
    return seq.reset_index(drop=True)


# Count violating triples and total excess
def tz_excess(schedule):
    triples = excess = 0
    for t in teams:
        seq = team_sequence(t, schedule)
        for k in range(len(seq) - 2):
            tz1, tz2, tz3 = (arena_tz[seq.iloc[k+i]["Arena"]] for i in range(3))
            jump = abs(tz2 - tz1) + abs(tz3 - tz2)
            if jump >= 4:
                triples += 1
                excess += jump - 3
    return triples, excess


orig_triples, orig_excess = tz_excess(games)
print(f"Original schedule: {orig_triples} violating triples, {orig_excess} hours of total excess.")

Original schedule: 22 violating triples, 33 hours of total excess.


### Use IIS to find constraints that cannot be satisfied:

We first build the model with the **hard** constraint $p_{12} + p_{23} \le 3$ for every team and every consecutive triple of game dates. If Gurobi returns infeasible, we use `computeIIS()` to extract an **Irreducible Inconsistent Subsystem** — the smallest set of constraints that cannot be satisfied simultaneously. The IIS pinpoints *where* the conflict lives: which teams, which triples, which (e)–(h) constraints are pulling against the timezone rule.

In [ ]:
def tz_on(x, t, d):
    # Linear expression for the timezone of team t on date d, given decision vars x
    home_term = team_tz[t] * gp.quicksum(x[t, j, d] for j in teams if j != t)
    away_term = gp.quicksum(team_tz[j] * x[j, t, d] for j in teams if j != t)
    return home_term + away_term

m_hard, x_hard = base_model("Q3_hard")

triples = []  # remember (team, k, d1, d2, d3) for IIS interpretation
for t in teams:
    sched = sorted(home_dates[t] + away_dates[t])
    for k in range(len(sched) - 2):
        d1, d2, d3 = sched[k], sched[k+1], sched[k+2]
        delta12 = tz_on(x_hard, t, d2) - tz_on(x_hard, t, d1)
        delta23 = tz_on(x_hard, t, d3) - tz_on(x_hard, t, d2)

        p12 = m_hard.addVar(lb=0, ub=3, name=f"p12[{t},{k}]")
        p23 = m_hard.addVar(lb=0, ub=3, name=f"p23[{t},{k}]")
        m_hard.addConstr(p12 >=  delta12, name=f"abs12pos[{t},{k}]")
        m_hard.addConstr(p12 >= -delta12, name=f"abs12neg[{t},{k}]")
        m_hard.addConstr(p23 >=  delta23, name=f"abs23pos[{t},{k}]")
        m_hard.addConstr(p23 >= -delta23, name=f"abs23neg[{t},{k}]")
        m_hard.addConstr(p12 + p23 <= 3, name=f"tz[{t},{k}]")
        triples.append((t, k, d1, d2, d3))

m_hard.optimize()

if m_hard.Status == GRB.INFEASIBLE:
    print("CONCLUSION: No feasible schedule satisfying the timezone constraint exists.")
    m_hard.computeIIS()
    m_hard.write("q3_iis.ilp")

    iis_tz_triples = [c.ConstrName for c in m_hard.getConstrs()
                      if c.IISConstr and c.ConstrName.startswith("tz[")]
    iis_eh = [c.ConstrName for c in m_hard.getConstrs()
              if c.IISConstr and not c.ConstrName.startswith(("tz[", "abs"))]

    print(f"IIS contains {sum(c.IISConstr for c in m_hard.getConstrs())} constraints:")
    print(f"  - {len(iis_tz_triples)} timezone constraints")
    print(f"  - {len(iis_eh)} structural constraints from (e)\u2013(h)")
    print(f"\nTimezone constraints in the IIS:")
    for name in iis_tz_triples:
        print(f"  {name}")
else:
    print("CONCLUSION: Feasible schedule found \u2014 timezone constraint is satisfiable.")
    sched3_hard = schedule_from_x(x_hard)
    sched3_hard["Arena"] = sched3_hard["Home"].map(team_arena)
    sched3_hard["Date"] = pd.to_datetime(sched3_hard["Date"], format="%a, %b %d, %Y")
    sched3_hard.to_csv("schedule_q3_feasible.csv", index=False)
    print(sched3_hard.head(10).to_string(index=False))


Set parameter LogToConsole to value 0
Hard model is infeasible — computing IIS.

IIS contains 40 constraints:
  - 3 timezone constraints
  - 31 structural constraints from (e)–(h)

Timezone constraints in the IIS:
  tz[Golden State Warriors,2]
  tz[Golden State Warriors,4]
  tz[Golden State Warriors,10]


### Add Slack variables to find a feasible solution

The IIS shows that the timezone constraints in conflict come from a small number of teams whose home/away pattern, fixed by (e)–(h), forces them into a coast-spanning triple at least once during the season. The (e)–(h) constraints are non-negotiable so the only way to obtain a feasible schedule is to relax the timezone rule itself.

We replace each $p_{12} + p_{23} \le 3$ with $p_{12} + p_{23} \le 3 + s_{t,k}$, $s_{t,k} \ge 0$, and minimize $\sum s_{t,k}$. The optimal $\sum s_{t,k}$ is a lower bound on the timezone burden any feasible schedule must accept — no schedule satisfying (e)–(h) can do better.

In [ ]:
m3, x3 = base_model("Q3_relaxed")
slack = {}

for t in teams:
    sched = sorted(home_dates[t] + away_dates[t])
    for k in range(len(sched) - 2):
        d1, d2, d3 = sched[k], sched[k+1], sched[k+2]
        delta12 = tz_on(x3, t, d2) - tz_on(x3, t, d1)
        delta23 = tz_on(x3, t, d3) - tz_on(x3, t, d2)

        p12 = m3.addVar(lb=0, ub=3, name=f"p12_{t}_{k}")
        p23 = m3.addVar(lb=0, ub=3, name=f"p23_{t}_{k}")
        s   = m3.addVar(lb=0, ub=3, name=f"s_{t}_{k}")
        m3.addConstr(p12 >=  delta12)
        m3.addConstr(p12 >= -delta12)
        m3.addConstr(p23 >=  delta23)
        m3.addConstr(p23 >= -delta23)
        m3.addConstr(p12 + p23 <= 3 + s)
        slack[t, k] = s

m3.setObjective(gp.quicksum(slack.values()), GRB.MINIMIZE)
m3.Params.TimeLimit = 120
m3.optimize()

opt_excess = round(m3.ObjVal)
print(f"Min total excess: {opt_excess} hours (original schedule: {orig_excess} hours).\n")
print(f"{'Team':28s} {'triples':>9} {'excess (hrs)':>14}")
for t in teams:
    n = sum(1 for (tt, _), v in slack.items() if tt == t and v.X > 1e-4)
    hrs = sum(v.X for (tt, _), v in slack.items() if tt == t)
    if hrs > 1e-4:
        print(f"{t:28s} {n:>9d} {hrs:>14.1f}")

# Save the Q3 best-possible schedule as a required deliverable
sched3 = schedule_from_x(x3)
sched3["Arena"] = sched3["Home"].map(team_arena)
sched3["Date"] = pd.to_datetime(sched3["Date"], format="%a, %b %d, %Y")
sched3.to_csv("schedule_q3_relaxed.csv", index=False)
q3_triples, q3_excess = tz_excess(sched3)
print(f"\nQ3 relaxed schedule saved \u2192 schedule_q3_relaxed.csv")
print(f"Tz violations: {q3_triples} triples, {q3_excess} hrs "
      f"(orig: {orig_triples} triples, {orig_excess} hrs)")


Set parameter LogToConsole to value 0
Min total excess: 27 hours (original schedule: 33 hours).

Team                           triples   excess (hrs)
Boston Celtics                       1            1.0
Brooklyn Nets                        1            1.0
Cleveland Cavaliers                  2            2.0
Golden State Warriors                5           10.0
Los Angeles Lakers                   3            3.0
Milwaukee Bucks                      1            1.0
New York Knicks                      2            2.0
Phoenix Suns                         3            4.0
Toronto Raptors                      1            3.0


## Question 4

We compare two objectives to improve the current schedule:

1. **Min total travel:** $\min \sum_t \mathrm{travel}(t)$
2. **Min max deviation from the average:** $\min \delta$ s.t. $|\mathrm{travel}(t) - \bar D| \le \delta$

Travel for a team between consecutive game dates breaks into four cases:

| from \ to | home | away |
|---|---|---|
| **home** | 0 | dist(team's arena, host's arena) — linear in $x$ |
| **away** | dist(host's arena, team's arena) — linear in $x$ | dist(host₁, host₂) — bilinear, linearized |

For away-to-away, we introduce $w \in \{0,1\}$ for each (host₁, host₂) pair with the standard
linearization $w \le x_1$, $w \le x_2$, $w \ge x_1 + x_2 - 1$, so $w = x_1 \cdot x_2$.

In [ ]:
arena_coords = {
    "TD Garden": (42.3662, -71.0621), "Madison Square Garden": (40.7505, -73.9934),
    "Barclays Center": (40.6826, -73.9754), "Wells Fargo Center": (39.9012, -75.1720),
    "Scotiabank Arena": (43.6435, -79.3791), "State Farm Arena": (33.7573, -84.3963),
    "Rocket Mortgage FieldHouse": (41.4965, -81.6882), "Kaseya Center": (25.7814, -80.1870),
    "United Center": (41.8807, -87.6742), "Fiserv Forum": (43.0451, -87.9172),
    "Toyota Center": (29.7508, -95.3621), "American Airlines Center": (32.7905, -96.8103),
    "Ball Arena": (39.7487, -105.0077), "Footprint Center": (33.4457, -112.0712),
    "Chase Center": (37.7680, -122.3877), "Crypto.com Arena": (34.0430, -118.2673),
}

def haversine(a, b):
    R = 3958.8  # earth radius in miles
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    h = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(h))

dist = {(a, b): haversine(arena_coords[a], arena_coords[b])
        for a in arena_coords for b in arena_coords}


def schedule_travel(schedule):
    out = {}
    for t in teams:
        seq = team_sequence(t, schedule)
        out[t] = sum(dist[seq.iloc[k]["Arena"], seq.iloc[k+1]["Arena"]]
                     for k in range(len(seq) - 1))
    return out


orig_travel = schedule_travel(games)
print(f"Original total travel: {sum(orig_travel.values()):,.0f} miles "
      f"(max {max(orig_travel.values()):,.0f}, avg {sum(orig_travel.values())/16:,.0f}).")

Original total travel: 200,087 miles (max 20,400, avg 12,505).


In [ ]:
def travel_model(name):
    m, x = base_model(name)
    travel = {t: gp.LinExpr() for t in teams}

    for t in teams:
        sched = sorted(home_dates[t] + away_dates[t])
        a_t = team_arena[t]

        for k in range(len(sched) - 1):
            d1, d2 = sched[k], sched[k+1]
            d1_home = d1 in home_dates[t]
            d2_home = d2 in home_dates[t]

            if d1_home and d2_home:
                continue                                         # HH: 0

            if d1_home and not d2_home:                          # HA
                travel[t] += gp.quicksum(
                    dist[a_t, team_arena[h]] * x[h, t, d2]
                    for h in teams if h != t
                )
                continue

            if not d1_home and d2_home:                          # AH
                travel[t] += gp.quicksum(
                    dist[team_arena[h], a_t] * x[h, t, d1]
                    for h in teams if h != t
                )
                continue

            # AA: linearize x[h1,t,d1] * x[h2,t,d2]
            for h1 in teams:
                if h1 == t: continue
                for h2 in teams:
                    if h2 == t or h1 == h2: continue
                    w = m.addVar(vtype=GRB.BINARY, name=f"w_{t}_{k}_{h1}_{h2}")
                    m.addConstr(w <= x[h1, t, d1])
                    m.addConstr(w <= x[h2, t, d2])
                    m.addConstr(w >= x[h1, t, d1] + x[h2, t, d2] - 1)
                    travel[t] += dist[team_arena[h1], team_arena[h2]] * w

    return m, x, travel


def report_travel(label, opt_travel):
    print(f"\n{label}")
    print(f"  total {sum(opt_travel.values()):,.0f}  "
          f"max {max(opt_travel.values()):,.0f}  "
          f"avg {sum(opt_travel.values())/16:,.0f}")
    print(f"  {'Team':28s} {'orig':>8} {'opt':>8} {'Δ':>8}")
    for t in sorted(teams, key=lambda t: -orig_travel[t]):
        d = opt_travel[t] - orig_travel[t]
        print(f"  {t:28s} {orig_travel[t]:>8,.0f} {opt_travel[t]:>8,.0f} {d:>+8,.0f}")

### Method 1 — Minimize total travel

In [ ]:
m1, x1, travel1 = travel_model("Q4_min_total")
m1.setObjective(gp.quicksum(travel1.values()), GRB.MINIMIZE)
m1.Params.TimeLimit = 300
m1.optimize()

print(f"Status {m1.Status}, gap {m1.MIPGap*100:.2f}%, total = {m1.ObjVal:,.0f} mi "
      f"(orig {sum(orig_travel.values()):,.0f}).")

sched1 = schedule_from_x(x1)
sched1["Arena"] = sched1["Home"].map(team_arena)
sched1["Date"] = pd.to_datetime(sched1["Date"], format="%a, %b %d, %Y")
sched1.to_csv("schedule_min_total.csv", index=False)

opt_travel_1 = schedule_travel(sched1)
report_travel("Method 1 (min total)", opt_travel_1)

Set parameter LogToConsole to value 0
Status 2, gap 0.00%, total = 192,082 mi (orig 200,087).

Method 1 (min total)
  total 192,082  max 20,517  avg 12,005
  Team                             orig      opt        Δ
  Golden State Warriors          20,400   20,517     +117
  Phoenix Suns                   16,730   15,971     -759
  Los Angeles Lakers             16,166   15,199     -966
  Boston Celtics                 15,756   13,520   -2,237
  New York Knicks                15,669   14,637   -1,031
  Philadelphia 76ers             13,977   12,864   -1,114
  Dallas Mavericks               13,573   13,311     -261
  Miami Heat                     12,798   12,503     -295
  Brooklyn Nets                  12,385   13,630   +1,245
  Toronto Raptors                10,789   12,134   +1,345
  Denver Nuggets                  9,922    8,756   -1,166
  Cleveland Cavaliers             9,688    8,320   -1,368
  Houston Rockets                 9,204    8,528     -676
  Milwaukee Bucks               

In [ ]:
print(f"Method 1: total travel "
      f"{sum(orig_travel.values()):,.0f} \u2192 {sum(opt_travel_1.values()):,.0f} mi "
      f"({(1 - sum(opt_travel_1.values())/sum(orig_travel.values()))*100:.1f}% reduction).")

m1_triples, m1_excess = tz_excess(sched1)
print(f"Method 1 tz violations: {m1_triples} triples, {m1_excess} hrs "
      f"(orig: {orig_triples} triples, {orig_excess} hrs)")


Method 1: total travel 200,087 → 192,082 mi (4.0% reduction).


### Method 2 — Minimize max deviation from league-average travel

Let $\mathrm{travel}(t)$ be team $t$'s total travel and $\bar D = \frac{1}{|T|} \sum_t \mathrm{travel}(t)$.
Minimize $\delta$ subject to
$\delta \ge \mathrm{travel}(t) - \bar D$ and $\delta \ge \bar D - \mathrm{travel}(t)$ for all $t$.

In [ ]:
m2, x2, travel2 = travel_model("Q4_min_dev")

T = {t: m2.addVar(lb=0, name=f"T_{t}") for t in teams}
for t in teams:
    m2.addConstr(T[t] == travel2[t])

avg = gp.quicksum(T.values()) / len(teams)
delta = m2.addVar(lb=0, name="delta")
for t in teams:
    m2.addConstr(delta >= T[t] - avg)
    m2.addConstr(delta >= avg - T[t])

m2.setObjective(delta, GRB.MINIMIZE)
m2.Params.TimeLimit = 300
m2.optimize()

print(f"Status {m2.Status}, gap {m2.MIPGap*100:.2f}%, "
      f"delta = {m2.ObjVal:,.0f}, league avg = {sum(T[t].X for t in teams)/16:,.0f}.")

sched2 = schedule_from_x(x2)
sched2["Arena"] = sched2["Home"].map(team_arena)
sched2["Date"] = pd.to_datetime(sched2["Date"], format="%a, %b %d, %Y")
sched2.to_csv("schedule_min_dev.csv", index=False)

opt_travel_2 = schedule_travel(sched2)
report_travel("Method 2 (min max-dev)", opt_travel_2)

Set parameter LogToConsole to value 0
Status 2, gap 0.00%, delta = 5,000, league avg = 13,140.

Method 2 (min max-dev)
  total 210,232  max 18,140  avg 13,140
  Team                             orig      opt        Δ
  Golden State Warriors          20,400   18,045   -2,355
  Phoenix Suns                   16,730   15,532   -1,198
  Los Angeles Lakers             16,166   14,909   -1,257
  Boston Celtics                 15,756   18,140   +2,384
  New York Knicks                15,669   17,681   +2,013
  Philadelphia 76ers             13,977   14,614     +637
  Dallas Mavericks               13,573   13,390     -183
  Miami Heat                     12,798   12,986     +188
  Brooklyn Nets                  12,385   13,881   +1,495
  Toronto Raptors                10,789   11,665     +876
  Denver Nuggets                  9,922    9,376     -546
  Cleveland Cavaliers             9,688   13,727   +4,039
  Houston Rockets                 9,204   11,274   +2,070
  Milwaukee Bucks            

In [ ]:
print(f"Method 2: max deviation "
      f"{max(abs(orig_travel[t] - sum(orig_travel.values())/16) for t in teams):,.0f} "
      f"\u2192 {m2.ObjVal:,.0f} mi.")

m2_triples, m2_excess = tz_excess(sched2)
print(f"Method 2 tz violations: {m2_triples} triples, {m2_excess} hrs "
      f"(orig: {orig_triples} triples, {orig_excess} hrs)")


Method 2: max deviation 7,895 → 5,000 mi.


### Method 3 — Minimize total travel subject to minimum timezone excess

We extend the Q4 travel model with Q3’s slack variables and add one budget constraint:
$$\sum_{t,k} s_{t,k} \le \texttt{opt\_excess}$$
This pins the total timezone excess to its proven lower bound (from Q3), so the schedule
is as timezone-friendly as structurally possible. Subject to that, we minimize total travel.

In [ ]:
m3t, x3t, travel3 = travel_model("Q4_min_total_tz")
slack3 = {}

for t in teams:
    sgames = sorted(home_dates[t] + away_dates[t])
    for k in range(len(sgames) - 2):
        d1, d2, d3 = sgames[k], sgames[k+1], sgames[k+2]
        delta12 = tz_on(x3t, t, d2) - tz_on(x3t, t, d1)
        delta23 = tz_on(x3t, t, d3) - tz_on(x3t, t, d2)

        p12 = m3t.addVar(lb=0, ub=3, name=f"p12_{t}_{k}")
        p23 = m3t.addVar(lb=0, ub=3, name=f"p23_{t}_{k}")
        s   = m3t.addVar(lb=0, ub=3, name=f"s_{t}_{k}")
        m3t.addConstr(p12 >=  delta12)
        m3t.addConstr(p12 >= -delta12)
        m3t.addConstr(p23 >=  delta23)
        m3t.addConstr(p23 >= -delta23)
        m3t.addConstr(p12 + p23 <= 3 + s)
        slack3[t, k] = s

# Lock tz excess to the proven minimum from Q3
m3t.addConstr(gp.quicksum(slack3.values()) <= opt_excess, name="tz_budget")

m3t.setObjective(gp.quicksum(travel3.values()), GRB.MINIMIZE)
m3t.Params.TimeLimit = 300
m3t.optimize()

print(f"Status {m3t.Status}, gap {m3t.MIPGap*100:.2f}%, total = {m3t.ObjVal:,.0f} mi "
      f"(orig {sum(orig_travel.values()):,.0f}).")

sched3t = schedule_from_x(x3t)
sched3t["Arena"] = sched3t["Home"].map(team_arena)
sched3t["Date"]  = pd.to_datetime(sched3t["Date"], format="%a, %b %d, %Y")
sched3t.to_csv("schedule_min_total_tz.csv", index=False)

opt_travel_3 = schedule_travel(sched3t)
report_travel("Method 3 (min total, tz-constrained)", opt_travel_3)

m3t_triples, m3t_excess = tz_excess(sched3t)
print(f"Method 3 tz violations: {m3t_triples} triples, {m3t_excess} hrs "
      f"(orig: {orig_triples} triples, {orig_excess} hrs)")


## Summary Comparison

Side-by-side comparison of all schedules across travel distance and timezone metrics.

In [ ]:
all_scheds = [
    ("Original",             games,   orig_travel),
    ("Q3 best (min tz)",     sched3,  schedule_travel(sched3)),
    ("Q4 M1 (min total)",    sched1,  opt_travel_1),
    ("Q4 M2 (min max-dev)",  sched2,  opt_travel_2),
    ("Q4 M3 (min total+tz)", sched3t, opt_travel_3),
]

rows = []
for name, sch, trav in all_scheds:
    trips, excess = tz_excess(sch)
    rows.append({
        "Schedule":        name,
        "Total (mi)":      f"{sum(trav.values()):,.0f}",
        "Max (mi)":        f"{max(trav.values()):,.0f}",
        "Avg (mi)":        f"{sum(trav.values())/len(teams):,.0f}",
        "tz triples":      trips,
        "tz excess (hrs)": excess,
    })

print(pd.DataFrame(rows).set_index("Schedule").to_string())


NameError: name 'games' is not defined